<a href="https://www.kaggle.com/code/prakashramsamy93/drive-behavior-prediction-using-ml-algorithms?scriptVersionId=348189270" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

<span style="font-size: 40px; font-weight: bold;">Drive behavior prediction using ML algorithms:</span>

# **Introduction**

Road safety is strongly influenced by driver behavior such as aggressive, distracted and unsafe driving can increase the likelihood of accidents and other hazardous situations. With the increasing availability of vehicle sensors and telematics data, machine learning can be used to analyze driving patterns and automatically identify potentially risky behavior.

The objective of this project is to develop a **Machine learning-based Driver Behavior Analysis system** that can classify driving behavior into different categories based on vehicle telemetry data.The dataset contains information related to driving characteristics such as **speed, acceleration, braking intensity, steering variability, lane deviation, following distance, reaction time, and phone usage indicators**. These features provide measurable information about how a driver operates a vehicle.

The ultimate goal is to demonstrate how data-driven techniques can assist in **early identification of risky driving patterns**, potentially supporting applications such as driver monitoring systems, road-safety solutions, fleet management, and intelligent transportation systems.

# **Dataset description**

The Vehicle Telemetry for Driver Behavior Analysis dataset contains vehicle and driving-related measurements used to analyze and classify driver behavior. It includes features such as speed, acceleration, braking intensity, steering variability, lane deviation, headway distance, reaction time, and phone usage.

The target variable categorizes driving behavior into three classes: Safe, Aggressive, and Distracted. This dataset can be used to develop machine learning classification models for identifying different driving patterns and supporting driver-safety and monitoring applications.

Dataset Source: **[Vehicle Telemetry for Driver Behavior Analysis](https://www.kaggle.com/datasets/sonalshinde123/vehicle-telemetry-for-driver-behavior-analysis)**

*Note : This dataset was synthetically generated and may not reflect real-world data*

**Loading necessary libraries**

In [ ]:
#importing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

 **Loading dataset**

In [ ]:
#loading dataset
import glob

def load_dataset():
    return pd.read_csv(
        glob.glob("/kaggle/input/**/Driver_Behavior.csv", recursive=True)[0]
    )

df = load_dataset()

print("Dataset loaded successfully!")

**Understanding the dataset**

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
#checking missig values
df.isnull().sum()

In [ ]:
#checking duplicates
print("Duplicate rows:", df.duplicated().sum())


In [ ]:
print(df.columns.tolist())

# **Exploratory Data Analysis**

**Driver behavior distribution**

In [ ]:
#plotting driver behaviors
plt.figure(figsize=(7, 5))

ax = sns.countplot(
    data=df,
    x='behavior_label',
    hue='behavior_label',
    palette='Set2',
    legend=False
)

plt.title("Distribution of Driver Behaviors")
plt.xlabel("Driver Behavior")
plt.ylabel("Number of Observations")

# Add counts
for container in ax.containers:
    ax.bar_label(container)

plt.show()


In [ ]:
behavior_pct = (
    df['behavior_label']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(behavior_pct)

**Outlier distribution**

In [ ]:
#tabulating outliers in percentage

numeric_cols = df.select_dtypes(include='number').columns

percentage_results = []

for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower) |
        (df[col] > upper)
    ]

    for behavior in df['behavior_label'].unique():

        total_behavior = (
            df['behavior_label'] == behavior
        ).sum()

        behavior_outliers = (
            outliers['behavior_label'] == behavior
        ).sum()

        percentage = (
            behavior_outliers / total_behavior * 100
        )

        percentage_results.append({
            'Feature': col,
            'Behavior': behavior,
            'Outlier %': round(percentage, 2)
        })

outlier_percentage_df = pd.DataFrame(
    percentage_results
)

display(outlier_percentage_df)

**Potential Outliers visual**

In [ ]:
#plotting outlier visuals
plt.figure(figsize=(14, 7))

sns.barplot(
    data=outlier_percentage_df,
    x='Feature',
    y='Outlier %',
    hue='Behavior',
    palette='Set2'
)

plt.title(
    'Percentage of Potential Outliers by Driver Behavior'
)

plt.xlabel('Telemetry Feature')
plt.ylabel('Outliers (%)')

plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()


The outlier percentage analysis reveals that Aggressive and Distracted behaviors contain a marginal share of extreme telemetry observations compared with Safe driving. Aggressive driving is characterized by extreme values in speed, acceleration and steering angle, whereas Distracted driving shows relatively extreme observations in lane deviation and steering angle.

he detected outliers are retained for further analysis because they may represent meaningful risky-driving events rather than data errors.

**Multivariate analysis**

In [ ]:
#performing multi variate analysis
important_features = [
    'speed_kmph',
    'accel_x',
    'accel_y',
    'brake_pressure',
    'steering_angle',
    'lane_deviation',
    'reaction_time'
]
sns.pairplot(
    df[important_features + ['behavior_label']],
    hue='behavior_label',
    palette='Set2',
    diag_kind='hist',
    corner=True
)
plt.show()

The pairplot shows that reaction time and brake pressure provide better separation between driver behaviors, while acceleration and steering angle have substantial overlap. Lane deviation and lateral acceleration show moderate separation. Overall, multiple features need to be considered together to accurately distinguish driver behaviors.

# **Dataset split**

**Feature/Target separation**

In [ ]:
# Features
features = [
    'speed_kmph',
    'accel_x',
    'accel_y',
    'brake_pressure',
    'steering_angle',
    'throttle',
    'lane_deviation',
    'phone_usage',
    'headway_distance',
    'reaction_time'
]

# Input features
X = df[features]

# Target variable
y = df['behavior_label']

print("Features shape:", X.shape)
print("Target shape:", y.shape)


**Train-Test Split**

In [ ]:
#splitting training and test dataset
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


# **Model Development & Evaluation**

Import necessary libraries

In [ ]:
#import necessary libraries
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)


**Logistic Regression**

In [ ]:
#LR model
lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])


In [ ]:
#train the model
lr.fit(X_train, y_train)


In [ ]:
#predict the model
lr_pred = lr.predict(X_test)

In [ ]:
#Evaluation 
print("Logistic Regression Accuracy:",
      accuracy_score(y_test, lr_pred))


In [ ]:
print(
    classification_report(
        y_test,
        lr_pred
    )
)


**Decision Tree**

In [ ]:
#Decision tree model
dt = DecisionTreeClassifier(
    random_state=42
)


In [ ]:
#train the model
dt.fit(X_train, y_train)


In [ ]:
#prediction
dt_pred = dt.predict(X_test)


In [ ]:
#evaluation
print("Decision Tree Accuracy:",
      accuracy_score(y_test, dt_pred))

print(
    classification_report(
        y_test,
        dt_pred
    )
)


**Random Forest**

In [ ]:
#Random Forest model
rf = RandomForestClassifier(
    random_state=42
)


In [ ]:
#train the model
rf.fit(X_train, y_train)


In [ ]:
#predict the model
rf_pred = rf.predict(X_test)


In [ ]:
#evaluation
print("Random Forest Accuracy:",
      accuracy_score(y_test, rf_pred))

print(
    classification_report(
        y_test,
        rf_pred
    )
)


**All models comparison**

In [ ]:
#comparing all models 
baseline_results = []

models_predictions = {
    'Logistic Regression': lr_pred,
    'Decision Tree': dt_pred,
    'Random Forest': rf_pred
}

for model_name, predictions in models_predictions.items():

    baseline_results.append({
        'Model': model_name,
        'Accuracy': accuracy_score(
            y_test,
            predictions
        ),
        'Precision': precision_score(
            y_test,
            predictions,
            average='weighted'
        ),
        'Recall': recall_score(
            y_test,
            predictions,
            average='weighted'
        ),
        'F1 Score': f1_score(
            y_test,
            predictions,
            average='weighted'
        )
    })

baseline_df = pd.DataFrame(baseline_results)

display(
    baseline_df.round(4)
)


**Decision Tree analysis**

In [ ]:
#analysizing Decision tree
print("Depth:", dt.get_depth())
print("Leaves:", dt.get_n_leaves())


The Decision Tree was examined first to understand the reason behind the perfect performance of the baseline classification models. The model achieved 100% accuracy with a tree depth of only 2 and three terminal leaves, indicating that the classification does not require a highly complex decision structure.

In [ ]:
from sklearn.tree import export_text

print(
    export_text(
        dt,
        feature_names=features
    )
)


The Decision Tree classifies driver behavior primarily using reaction time and phone usage. Drivers with a reaction time of 0.70 or below are classified as Aggressive. Among drivers with a reaction time above 0.70, those with no significant phone usage are classified as Safe, while those with phone usage are classified as Distracted.

**Decision Tree Feature Importance**

In [ ]:
#analysizing Decision tree feature importance
importance = pd.DataFrame({
    'Feature': features,
    'Importance': dt.feature_importances_
}).sort_values('Importance', ascending=False)

display(importance)


The feature importance analysis showed that reaction_time and phone_usage were the two most influential features, each contributing 50% importance. All other features had zero importance in the Decision Tree, indicating that these two variables primarily determine the classification of driver behavior in this dataset.

**Random Forest Feature Importance**

In [ ]:
#analysizing Random Forest feature importance
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

plt.figure(figsize=(8,5))

sns.barplot(
    x="Importance",
    y="Feature",
    data=importance_df
)

plt.title("Feature Importance – Random Forest")
plt.show()


# **Conclusion**

This notebook explored driver behavior using vehicle telemetry data through exploratory data analysis and machine learning classification. The EDA examined feature distributions, relationships between variables, outliers, and the degree of overlap among different driver behavior classes.

Three classification models—Logistic Regression, Decision Tree, and Random Forest—were developed as baseline models. All three achieved perfect performance on the test set, with accuracy, precision, recall, and F1-score of 1.00.

Further analysis of the Decision Tree showed that this performance could be explained using a very simple decision structure with a depth of 2 and three leaves. The model relied primarily on reaction_time and phone_usage. In particular, reaction time provided very strong separation between the behavior classes, with a single-feature Decision Tree achieving 100% accuracy.

The Random Forest analysis was used to further evaluate model performance and identify important predictive features. The results indicate that the driver behavior classes in this dataset are highly separable using the available telemetry variables.

**Overall, the analysis demonstrates that vehicle telemetry can effectively distinguish different driver behaviors.**